## Tools

##### Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of: 1) A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema) 2) A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Why do parrots talk?")
response.content

'Parrots talk primarily because of their unique biological makeup, social nature, and cognitive abilities. It is not that they "understand" language in the same way humans do, but rather that they have mastered the mechanics of vocal mimicry.\n\nHere’s a breakdown of why and how parrots talk:\n\n### 1. Biological Adaptation: The Vocal Sacs\nUnlike most birds, parrots have a specialized vocal organ called the **syrinx**, located where the trachea (windpipe) splits into the bronchi. Parrots’ syrinxes are particularly complex and powerful, allowing them to produce a wide range of sounds, including the complex, resonant, and varied tones of human speech. They also have a unique **vocal sac** that acts like a resonating chamber, amplifying their voices.\n\n### 2. Social Bonding and Group Identity\nIn the wild, parrots are highly social animals that live in large, noisy flocks. Communication is essential for survival, coordination, and maintaining social bonds.\n- **Mimicry as Social Integra

In [2]:
from langchain.tools import tool

@tool 
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'dqeen5nzm', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.067864886, 'completion_tokens_details': None, 'prompt_time': 0.019550075, 'prompt_tokens_details': None, 'queue_time': 0.046290354, 'total_time': 0.087414961}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a13dfd-6b09-7443-a587-cb0b650e49a6-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'dqeen5nzm', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."